# 7.2 Face Inference Demo

Purpose:
- Demonstrate that the saved visual model can be loaded.
- Run inference on a single image.
- Output predicted class, probability, and confidence.
- Save a report-ready demo figure.

In [1]:
from pathlib import Path
import pandas as pd
import sys

sys.path.insert(0, str(Path.cwd().parent))

from rural_stroke_assist.config import (
    FACE_SPLIT_MANIFEST_PATH,
    FACE_IMAGE_SIZE,
    FACE_TRIAL_003_MODEL_PATH,
    FACE_DEMO_PREDICTION_FIGURE_PATH,
)

from rural_stroke_assist.modeling.face_inference import (
    load_face_model,
    predict_face_image,
)

from rural_stroke_assist.modeling.demo_visualization import (
    save_face_prediction_demo_figure,
)

In [13]:
model = load_face_model(FACE_TRIAL_003_MODEL_PATH)

model.summary()

Model: "mobilenetv2_face_binary_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ multiply (Multiply)             │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,261,829 (8.63 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

 Optimizer params: 2,564 (10.02 KB)

In [14]:
face_split_df = pd.read_csv(FACE_SPLIT_MANIFEST_PATH)

demo_row = (
    face_split_df[
        (face_split_df["split"] == "test")
        & (face_split_df["class_label"] == "Stroke")
    ]
    .sample(1, random_state=40)
    .iloc[0]
)

demo_image_path = demo_row["path"]

demo_row

path           C:\Users\Asus\Downloads\Uni_work\Grad-Project\...
file_name                                           img_0403.jpg
class_label                                               Stroke
extension                                                   .jpg
readable                                                    True
width                                                        156
height                                                       156
file_hash      d80f98f5fc95965a4a2bd5b977868a9c4bf1d110cadf38...
is_clean                                                    True
split                                                       test
Name: 1500, dtype: object

In [15]:
prediction = predict_face_image(
    model=model,
    image_path=demo_image_path,
    image_size=FACE_IMAGE_SIZE,
    threshold=0.5,
)

prediction

{'image_path': 'C:\\Users\\Asus\\Downloads\\Uni_work\\Grad-Project\\rural-stroke-assist\\data\\raw\\face_stroke_images\\Annotated stroke and non stroke Dataset\\Stroke\\img_0403.jpg',
 'stroke_probability': 0.701987087726593,
 'predicted_label_encoded': 1,
 'predicted_label': 'Stroke',
 'confidence': 0.701987087726593,
 'threshold': 0.5}

In [16]:
{
    "true_label": demo_row["class_label"],
    "predicted_label": prediction["predicted_label"],
    "stroke_probability": prediction["stroke_probability"],
    "confidence": prediction["confidence"],
}

{'true_label': 'Stroke',
 'predicted_label': 'Stroke',
 'stroke_probability': 0.701987087726593,
 'confidence': 0.701987087726593}

In [12]:
save_face_prediction_demo_figure(
    image_path=demo_image_path,
    prediction=prediction,
    output_path=FACE_DEMO_PREDICTION_FIGURE_PATH,
)

FACE_DEMO_PREDICTION_FIGURE_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/reports/figures/experiments/face_demo_prediction.png')